## Descriptive Statistics

This section provides an overview of the Business Analyst job market dataset. The analysis focuses on job posting characteristics, geographic and industry distributions, education and experience requirements, salary information, and technical skill requirements.

## 4.1 Import Libraries and Load Data

The dataset can be loaded directly from the Raw GitHub URL. Replace the placeholder below with the Raw URL of `pilot_data_cleaned.xlsx`.

In [2]:
import pandas as pd

df = pd.read_excel("../data/cleaned/pilot_data_cleaned.xlsx")

print(df.head())

   Job_id        Country           City  \
0  BA0001  United States    Atlanta, GA   
1  BA0002  United States     Reston, VA   
2  BA0003  United States      Miami, FL   
3  BA0004  United States     Boston, MA   
4  BA0005  United States  Las Vegas, NV   

                                           Job_Title             Company_Name  \
0                             Staff Business Analyst                   TriNet   
1  Business Analyst (ServiceNow) - (High Level Cl...                      ICF   
2          Business Analyst, Fuse Billing Operations                   Amazon   
3                       Systems Business Analyst Sr.  Brown Brothers Harriman   
4                                Sr Business Analyst    Las Vegas Sands Corp.   

     Industry    Source                                            Job_Url  \
0  Technology  LinkedIn  https://www.linkedin.com/jobs/view/staff-busin...   
1  Technology  LinkedIn  https://www.linkedin.com/jobs/view/business-an...   
2  Technology  Linke

## 4.2 Basic Dataset Information

This section reports the number of job postings, variables, countries, industries, job titles, and companies represented in the dataset.

In [3]:
print("Number of job postings:", df.shape[0])
print("Number of variables:", df.shape[1])
print("Number of countries:", df["Country"].nunique())
print("Number of industries:", df["Industry"].nunique())
print("Number of unique job titles:", df["Job_Title"].nunique())
print("Number of unique companies:", df["Company_Name"].nunique())


Number of job postings: 100
Number of variables: 25
Number of countries: 4
Number of industries: 6
Number of unique job titles: 68
Number of unique companies: 83


In [4]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
df.info()

Column names:
['Job_id', 'Country', 'City', 'Job_Title', 'Company_Name', 'Industry', 'Source', 'Job_Url', 'Date_Posted', 'Date_Collected', 'Salary_Min', 'Salary_Max', 'Salary_Currency', 'Experience_Min', 'Experience_Max', 'Education', 'Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'AI_tools', 'JD_text', 'Collector']

Data types:
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 25 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Job_id           100 non-null    str           
 1   Country          100 non-null    str           
 2   City             100 non-null    str           
 3   Job_Title        100 non-null    str           
 4   Company_Name     100 non-null    str           
 5   Industry         100 non-null    str           
 6   Source           100 non-null    str           
 7   Job_Url          100 non-null    str           
 8   Date_Posted      100 non-null    d

## 4.3 Job Postings by Country

The following table shows the number and percentage of Business Analyst job postings by country.

In [5]:
country_summary = (
    df["Country"]
    .value_counts(dropna=False)
    .rename_axis("Country")
    .reset_index(name="Job_Postings")
)

country_summary["Percentage"] = (
    country_summary["Job_Postings"] / len(df) * 100
).round(1)

country_summary

,Country,Job_Postings,Percentage
0,United States,35,35.0
1,United Kingdom,25,25.0
2,Singapore,20,20.0
3,Hong Kong,20,20.0


## 4.4 Job Postings by Industry

This section summarizes the industries represented in the sample. Category consistency should be checked before drawing conclusions.

In [6]:
industry_summary = (
    df["Industry"]
    .value_counts(dropna=False)
    .rename_axis("Industry")
    .reset_index(name="Job_Postings")
)

industry_summary["Percentage"] = (
    industry_summary["Job_Postings"] / len(df) * 100
).round(1)

industry_summary

,Industry,Job_Postings,Percentage
0,Technology,69,69.0
1,Financial Services,27,27.0
2,Industiral,1,1.0
3,Education,1,1.0
4,Financial services,1,1.0
5,Unknown,1,1.0


In [ ]:
# Check the original industry labels for possible inconsistencies
print(df["Industry"].value_counts(dropna=False))

## 4.5 Education Requirements

This section examines the educational qualifications reported in the job postings.

In [ ]:
education_summary = (
    df["Education"]
    .value_counts(dropna=False)
    .rename_axis("Education")
    .reset_index(name="Job_Postings")
)

education_summary["Percentage"] = (
    education_summary["Job_Postings"] / len(df) * 100
).round(1)

education_summary

## 4.6 Experience Requirements

Experience fields contain some non-numeric values such as `Not Specified`. These are converted to missing values so that numerical descriptive statistics can be calculated correctly.

In [ ]:
df["Experience_Min_num"] = pd.to_numeric(
    df["Experience_Min"], errors="coerce"
)

df["Experience_Max_num"] = pd.to_numeric(
    df["Experience_Max"], errors="coerce"
)

experience_summary = df[
    ["Experience_Min_num", "Experience_Max_num"]
].describe().T

experience_summary

In [ ]:
print("Median minimum experience:", df["Experience_Min_num"].median())
print("Median maximum experience:", df["Experience_Max_num"].median())

print("\nMissing rate of Experience_Min: "
      f"{df['Experience_Min_num'].isna().mean() * 100:.1f}%")

print("Missing rate of Experience_Max: "
      f"{df['Experience_Max_num'].isna().mean() * 100:.1f}%")

## 4.7 Salary Information

Salary values are reported in different currencies. Therefore, salary statistics are summarized within each currency rather than combining currencies directly.

In [ ]:
df["Salary_Min_num"] = pd.to_numeric(
    df["Salary_Min"], errors="coerce"
)

df["Salary_Max_num"] = pd.to_numeric(
    df["Salary_Max"], errors="coerce"
)

print("Salary currencies:")
display(df["Salary_Currency"].value_counts(dropna=False).to_frame("Job_Postings"))

In [ ]:
salary_summary = (
    df.groupby("Salary_Currency")
      [["Salary_Min_num", "Salary_Max_num"]]
      .agg(["count", "mean", "median", "min", "max"])
)

salary_summary

In [ ]:
salary_availability = pd.DataFrame({
    "Variable": ["Salary_Min", "Salary_Max"],
    "Available": [
        df["Salary_Min_num"].notna().sum(),
        df["Salary_Max_num"].notna().sum()
    ]
})

salary_availability["Missing"] = (
    len(df) - salary_availability["Available"]
)

salary_availability["Missing_Rate"] = (
    salary_availability["Missing"] / len(df) * 100
).round(1)

salary_availability

## 4.8 Technical Skill Requirements

The skill variables are coded as binary indicators, where 1 means that the skill is identified in the job posting and 0 means that it is not identified.

In [ ]:
skill_columns = [
    "Excel",
    "SQL",
    "Python",
    "R",
    "Tableau",
    "Power_BI",
    "AI_tools"
]

skill_summary = pd.DataFrame({
    "Skill": skill_columns,
    "Number_of_Jobs": [df[col].sum() for col in skill_columns]
})

skill_summary["Percentage"] = (
    skill_summary["Number_of_Jobs"] / len(df) * 100
).round(1)

skill_summary = skill_summary.sort_values(
    "Number_of_Jobs", ascending=False
).reset_index(drop=True)

skill_summary

## 4.9 Missing Value Summary

The following table identifies variables with missing observations and their missing-data rates.

In [ ]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(1)
})

missing_summary = (
    missing_summary[missing_summary["Missing_Count"] > 0]
    .sort_values("Missing_Count", ascending=False)
)

missing_summary

## 4.10 Key Statistics Summary

This summary highlights several key characteristics of the sampled Business Analyst job market.

In [ ]:
most_requested_skill = skill_summary.loc[
    skill_summary["Number_of_Jobs"].idxmax(), "Skill"
]

key_statistics = pd.DataFrame({
    "Metric": [
        "Total job postings",
        "Number of countries",
        "Number of industries",
        "Median minimum experience (years)",
        "Salary Min available",
        "Salary Max available",
        "Most frequently identified technical skill"
    ],
    "Value": [
        len(df),
        df["Country"].nunique(),
        df["Industry"].nunique(),
        df["Experience_Min_num"].median(),
        df["Salary_Min_num"].notna().sum(),
        df["Salary_Max_num"].notna().sum(),
        most_requested_skill
    ]
})

key_statistics

## 4.11 Initial Interpretation

The descriptive statistics provide a baseline view of the sampled Business Analyst labor market. The dataset covers multiple geographic markets and industries, while experience and technical-skill information is more consistently available than salary information. Because salary disclosure is incomplete and the dataset is based on a limited sample of job postings, salary comparisons should be interpreted cautiously. The next section uses visualization to examine geographic, industry, experience, salary, and skill patterns in greater detail.